# Lab 2 — Single-expiry options market making, local volatility, 10 options
Slides 22–24. $dS_t/S_t = \sigma_{loc}(t,S_t)\,dW_t$; implied vols $\sigma(K,T)$ constant over the session; marks $C_t(K,T) = C_{BS}(S_t,K,T-t,\sigma(K,T))$; quotes $C_t\pm\delta$ with per-option fills as in Lab 1. Delta-hedged book value (slide 23) with $\Gamma^\$ = \Gamma S^2$:
$$\Pi_\tau = \sum_i\int_0^\tau \delta^{b,i}_t dN^{b,i}_t+\delta^{a,i}_t dN^{a,i}_t \;+\; \tfrac12\sum_i\int_0^\tau\big(\sigma_t^2-\sigma^2(K_i,T)\big)\,Q^i_t\,\Gamma^{\$,i}_t\,dt,$$
objective $J(\delta)=\mathbb E[\Pi_\tau-\alpha\int|Q_t|^2dt-\beta|Q_\tau|^2]$ with $|Q|^2=\sum_i (Q^i)^2$.

Sign convention: a delta-hedged long option earns $\tfrac12(\sigma_{realized}^2-\sigma_{implied}^2)\Gamma^\$dt$ (slide 23 has the difference the other way round); the identity is checked numerically below. Time in trading days as in Lab 1, vols in daily units.

In [ ]:
import math, torch, numpy as np, matplotlib.pyplot as plt
from omm import *
torch.manual_seed(0)

S0, tau, N = 100.0, 1.0, 390; dt = tau / N
T = 21.0                                                                 # single expiry, 21 trading days (tau << T)
K = torch.linspace(90, 110, 10, device=DEV); n = len(K); m = torch.log(K / S0)
sig_imp = (0.20 - 0.30 * m + 0.50 * m ** 2) / math.sqrt(252)            # implied smile sigma(K,T)
sig_loc = lambda t, S: 0.22 / math.sqrt(252) * (S / S0) ** (-1.0)        # local vol sigma_loc(t,S)
sig_ref = 0.20 / math.sqrt(252)
lam_i = torch.full((n,), 20.0, device=DEV)
_, _, _, vega = bs_call(torch.tensor(S0, device=DEV), K, torch.tensor(T, device=DEV), sig_imp)
kappa_i = 1.5 / (0.01 * vega / math.sqrt(252))                           # 1/kappa_i = 0.67 vol points in price: comparable across strikes
alpha, beta = 1e-3, 1e-3
print(DEV, "FAST" if FAST else "full scale")

## Marks, simulation and the hedged P&L identity

In [ ]:
def marks(t, S):                                                        # BS price, delta, dollar gamma at the implied vols, (B,n)
    C, D, G, _ = bs_call(S[:, None], K, (T - t * torch.ones_like(S))[:, None], sig_imp)
    return C, D, G * S[:, None] ** 2

def simulate(policy, B, seed=None):
    g = torch.Generator(device=DEV)
    if seed is not None: g.manual_seed(seed)
    dW = math.sqrt(dt) * torch.randn(B, N, device=DEV, generator=g)
    S, Q = torch.full((B,), S0, device=DEV), torch.zeros(B, n, device=DEV)
    out = dict(rbar=[], logp=[], cost=[], dN=[], Q=[Q]); spread = mkt = run = 0.0
    for k in range(N):
        t = k * dt; sl = sig_loc(t, S); _, _, G = marks(t, S)
        d = policy(t, S, Q)                                                 # (B,n,2)
        lam_dt = lam_i[:, None] * torch.exp(-kappa_i[:, None] * d) * dt
        dN, lp = fills(lam_dt)
        drift = (0.5 * ((sl ** 2)[:, None] - sig_imp ** 2) * Q * G).sum(1) * dt   # hedged inventory P&L (slide 23)
        cost = alpha * (Q ** 2).sum(1) * dt
        out["rbar"].append((d * lam_dt).sum((1, 2))); out["logp"].append(lp.sum((1, 2)))
        out["cost"].append(cost - drift); out["dN"].append(dN)
        spread = spread + (d.detach() * dN).sum((1, 2)); mkt = mkt + drift; run = run + cost
        Q, S = Q + dN[..., 0] - dN[..., 1], S * torch.exp(sl * dW[:, k] - 0.5 * sl ** 2 * dt)
        out["Q"].append(Q)
    out = {k: torch.stack(v, 1) for k, v in out.items()}
    out.update(term=beta * (Q ** 2).sum(1), S=S, spread=spread, mkt=mkt, pnl=spread + mkt)
    out["J"] = out["pnl"] - run - out["term"]
    return out

myopic = lambda t, S, Q: (1 / kappa_i)[None, :, None].expand(len(S), n, 2)

In [ ]:
# C_tau - C_0 - int Delta dS  vs  1/2 int (sigma_loc^2 - sigma^2(K,T)) Gamma^$ dt, per unit option, averaged over paths
B = 4000; g = torch.Generator(device=DEV).manual_seed(3)
S = torch.full((B,), S0, device=DEV); lhs, rhs = -marks(0.0, S)[0], torch.zeros(B, n, device=DEV)
for k in range(N):
    t = k * dt; sl = sig_loc(t, S); _, D, G = marks(t, S)
    S_new = S * torch.exp(sl * math.sqrt(dt) * torch.randn(B, device=DEV, generator=g) - 0.5 * sl ** 2 * dt)
    lhs, rhs, S = lhs - D * (S_new - S)[:, None], rhs + 0.5 * ((sl ** 2)[:, None] - sig_imp ** 2) * G * dt, S_new
lhs = lhs + marks(tau, S)[0]
print("K, hedged P&L (mean), theta-gamma term (mean):\n", torch.stack([K, lhs.mean(0), rhs.mean(0)], 1).cpu().numpy().round(4))

## DSO (slide 12): $\delta^\theta_t = \tfrac1\kappa + \Phi(t,S_t,Q_t;\theta)\in\mathbb R^{n\times 2}$

In [ ]:
def feats(t, S, Q):
    return torch.cat([(t / tau * torch.ones_like(S))[:, None], (torch.log(S / S0) / sig_ref)[:, None], Q / 5.0], -1)

net = MLP(2 + n, 2 * n, width=128, zero_last=True).to(DEV)
dso_policy = lambda t, S, Q: 1 / kappa_i[:, None] + ckpt(net, feats(t, S, Q)).view(-1, n, 2)

def dso_step():
    o = simulate(dso_policy, B=scale(2048, 8))
    surr, J = dso_surrogate(o["rbar"], o["logp"], o["cost"], o["term"])
    return -surr, J

hist_dso = train(dso_step, net.parameters(), iters=scale(1500), lr=1e-3, clip=10.0, name="DSO")

## DGM (slide 24): $v(t,S_t,Q_t)$ under local volatility (Lucic & Tse, 2025)
$$\partial_t v + \tfrac12\sigma_{loc}^2S^2\partial_{SS}v + \tfrac12\sum_i(\sigma_{loc}^2-\sigma_i^2)\,Q^i\,\Gamma^{\$,i}(t,S) - \alpha|Q|^2 + \sum_i\tfrac{\lambda_i}{\kappa_i}\big(e^{\kappa_i\Delta^{b,i}v-1}+e^{\kappa_i\Delta^{a,i}v-1}\big)=0,\qquad v(\tau,S,Q)=-\beta|Q|^2,$$
with $\Delta^{b,i}v = v(t,S,Q+e_i)-v(t,S,Q)$, $\Delta^{a,i}v = v(t,S,Q-e_i)-v(t,S,Q)$ and $\delta^{b,i}=\tfrac1{\kappa_i}-\Delta^{b,i}v$, $\delta^{a,i}=\tfrac1{\kappa_i}-\Delta^{a,i}v$.

In [ ]:
Qmax = 15; E = torch.eye(n, device=DEV)
vnet = MLP(2 + n, 1, width=128, depth=3).to(DEV)
V = lambda t, S, Q: vnet(feats(t, S, Q)).squeeze(-1)

def increments(t, S, Q):                                                  # v, Delta^b v, Delta^a v  for all i, (B,n)
    v = V(t, S, Q); tt, SS = t.detach().repeat_interleave(n), S.detach().repeat_interleave(n)
    Vb, Va = V(tt, SS, (Q[:, None] + E).view(-1, n)).view(-1, n), V(tt, SS, (Q[:, None] - E).view(-1, n)).view(-1, n)
    return v, Vb - v[:, None], Va - v[:, None]

def hjb(t, S, Q):
    t, S = t.requires_grad_(True), S.requires_grad_(True)
    v, Db, Da = increments(t, S, Q)
    v_t, v_SS = grad(v, t), grad(grad(v, S), S)
    sl = sig_loc(t, S); _, _, G = marks(t, S)
    drift = (0.5 * ((sl ** 2)[:, None] - sig_imp ** 2) * Q * G).sum(1)
    H = (lam_i / kappa_i * (torch.exp(kappa_i * Db - 1) + torch.exp(kappa_i * Da - 1))).sum(1)
    return v_t + 0.5 * sl ** 2 * S ** 2 * v_SS + drift - alpha * (Q ** 2).sum(1) + H

def dgm_step(B=scale(1024, 4), w=10.0):
    t, S = tau * torch.rand(B, device=DEV), S0 * torch.exp(2 * sig_ref * torch.randn(B, device=DEV))
    Q = torch.randint(-Qmax, Qmax + 1, (B, n), device=DEV).float()
    loss = hjb(t, S, Q).pow(2).mean() + w * (V(torch.full_like(t, tau), S, Q) + beta * (Q ** 2).sum(1)).pow(2).mean()
    return loss, loss.item()

hist_dgm = train(dgm_step, vnet.parameters(), iters=scale(4000), lr=1e-3, name="DGM")

def dgm_policy(t, S, Q):
    _, Db, Da = increments(t * torch.ones_like(S), S, Q)
    return torch.stack([1 / kappa_i - Db, 1 / kappa_i - Da], -1)

## Comparison

In [ ]:
policies = {"myopic 1/k": myopic, "DSO": dso_policy, "DGM": dgm_policy}
with torch.no_grad():
    for i, (name, pol) in enumerate(policies.items()):
        o = simulate(pol, B=scale(8000, 8), seed=1)
        report(name, o["pnl"], o["J"], o["Q"][:, -1].norm(dim=1), o["dN"].sum((1, 2, 3)).mean().item(), header=(i == 0))

In [ ]:
Kn = K.cpu().numpy(); atm = n // 2
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
ax[0].plot(Kn, sig_imp.cpu().numpy() * math.sqrt(252), "o-", color=COL["mid"], label="implied  sigma(K,T)")
ax[0].axhline(sig_loc(0.0, torch.tensor(S0)).item() * math.sqrt(252), color="k", ls="--", label="local  sigma_loc(0,S0)")
ax[0].set_xlabel("strike K"); ax[0].set_title("where the vol arbitrage is", fontsize=10); ax[0].legend(fontsize=8)
with torch.no_grad():                                                     # quotes at Q = 0, t = 0, relative to the myopic 1/kappa_i
    S1, Q0 = torch.tensor([S0], device=DEV), torch.zeros(1, n, device=DEV)
    for name, pol, ls in [("DSO", dso_policy, "o"), ("DGM", dgm_policy, "--")]:
        r = (pol(0.0, S1, Q0)[0] * kappa_i[:, None]).cpu().numpy()
        ax[1].plot(Kn, r[:, 0], ls, color=COL["bid"], label=f"{name} bid"); ax[1].plot(Kn, r[:, 1], ls, color=COL["ask"], label=f"{name} ask")
    ax[1].axhline(1, color="gray", lw=0.5); ax[1].set_xlabel("strike K"); ax[1].set_title("kappa_i * delta^i at Q = 0, t = 0", fontsize=10); ax[1].legend(fontsize=7, ncol=2)
    qq = torch.arange(-15, 16, device=DEV).float(); Qa = torch.zeros(len(qq), n, device=DEV); Qa[:, atm] = qq   # ATM option vs its own inventory
    for name, pol, ls in [("DSO", dso_policy, "o"), ("DGM", dgm_policy, "--")]:
        r = (pol(0.0, torch.full_like(qq, S0), Qa)[:, atm] * kappa_i[atm]).cpu().numpy()
        ax[2].plot(qq.cpu(), r[:, 0], ls, color=COL["bid"], ms=3); ax[2].plot(qq.cpu(), r[:, 1], ls, color=COL["ask"], ms=3)
    ax[2].set_xlabel(f"inventory in the K = {Kn[atm]:.0f} option"); ax[2].set_title("kappa * delta for the ATM option (DSO dots, DGM dashed)", fontsize=10)
plt.tight_layout()

In [ ]:
# cross-inventory dependence: do the ATM quotes move when the inventory of another option changes?  (-> Lab 3)
with torch.no_grad():
    Qx = torch.zeros(len(qq), n, device=DEV); Qx[:, 0] = qq
    for name, pol in [("DSO", dso_policy), ("DGM", dgm_policy)]:
        r = pol(0.0, torch.full_like(qq, S0), Qx)[:, atm] * kappa_i[atm]
        print(f"{name}: ATM quotes vs inventory of the K = {Kn[0]:.0f} option, max |change| = {(r - r[15]).abs().max():.4f}  (in units of 1/kappa)")